# InfraPulse — train the defect classification head on Kaggle

Trains only a small head on top of a **frozen ImageNet-pretrained MobileNetV2**
backbone (generic backbone, allowed by the PS rules). No checkpoint pretrained
specifically on defect/damage data is used anywhere.

**Before running:**
1. In the Kaggle notebook editor, use *Add Input* to attach the 4 datasets listed below.
2. Settings → Internet → **On** (needed to download the ImageNet MobileNetV2 weights once).
3. Settings → Accelerator → GPU is optional (CPU is fine, this only trains a 2-layer head).
4. Run cells in order top to bottom (or Run All) — cell `2b` prints a mask pixel-value histogram and defines `SPALLING_MASK_VALUES`, which the collection cell depends on. Don't skip it.
5. At the end, download `head.pt` from the notebook's Output panel and copy it to `backend/app/models_weights/head.pt` in the repo.

**Datasets used (4 different formats — each handled by its own collector below, not one generic loader):**
- `cracked_tiles` ← [Ceramic Tiles Defects](https://www.kaggle.com/datasets/riffatsiddiqui/ceramic-tiles-defects-crackspotspinhole) — flat folder, filenames prefixed `crack.` / `spot.` / `pinhole.`
- `stagnant_water` ← [Stagnant Water and Wet Surface](https://www.kaggle.com/datasets/meraxes10/stagnantwaterdata) — YOLO format: `<name>.jpeg` + `<name>.txt` (bbox labels) + `classes.txt`
- `paint_peeling` ← [wall paint](https://www.kaggle.com/datasets/sri2511/wall-paint) — `worse_condtion` vs `good_condtion` subfolders (nested; note the dataset's own "condtion" typo)
- `spalling` ← [S2DS for concrete defects segmentation](https://www.kaggle.com/datasets/raidathmane/s2ds-for-concrete-defects-segmentation) — multi-class segmentation (background/crack/spalling/corrosion/efflorescence/vegetation/control point). We tried [corrosion & spalling, concrete defect segmentation](https://www.kaggle.com/datasets/raidathmane/corrosion-and-spalling-concrete-defect-segmentation) first, but its masks turned out to be binary (`{0: background, 255: any defect}`) with no way to separate spalling from corrosion — confirmed by actually running the histogram, not assumed. S2DS is multi-class so it should work, but its exact file-naming convention hasn't been verified yet either — that's what cell `2b` checks.


In [ ]:
import os
import glob

# 1) Inspect what Kaggle actually mounted for each dataset you attached.
# Run this, read the printed tree, then check the mask-inspection cell below
# before trusting the spalling collector.
ROOT = "/kaggle/input"
for dataset_dir in sorted(os.listdir(ROOT)):
    print(f"\n=== {dataset_dir} ===")
    base = os.path.join(ROOT, dataset_dir)
    for cur_root, dirs, files in os.walk(base):
        depth = cur_root[len(base):].count(os.sep)
        if depth > 4:
            continue
        indent = "  " * depth
        n_images = sum(1 for f in files if f.lower().endswith((".jpg", ".jpeg", ".png")))
        print(f"{indent}{os.path.basename(cur_root)}/  ({n_images} images, {len(dirs)} subdirs)")


def find_dataset_root(slug: str) -> str:
    """Locate a dataset's mounted folder by slug regardless of nesting depth
    under /kaggle/input (observed layouts have varied: flat /kaggle/input/<slug>
    vs /kaggle/input/datasets/<owner>/<slug>)."""
    matches = glob.glob(f"/kaggle/input/**/{slug}", recursive=True)
    matches = [m for m in matches if os.path.isdir(m)]
    if not matches:
        raise FileNotFoundError(f"no folder named '{slug}' found under /kaggle/input — check it's attached")
    return matches[0]

In [ ]:
# 2) Helpers for each dataset's actual format (based on the real Kaggle
# "Data Explorer" trees for these 4 datasets — not a generic ImageFolder
# loader, because none of these datasets share a layout).
IMG_EXTS = (".jpg", ".jpeg", ".png")


def collect_by_filename_prefix(root: str, prefix: str):
    """Ceramic Tiles Defects: flat folder, files named '<prefix>.<n>.jpg'."""
    pattern = os.path.join(root, "**", f"{prefix}.*")
    return [p for p in glob.glob(pattern, recursive=True) if p.lower().endswith(IMG_EXTS)]


def collect_by_folder_keyword(root: str, keyword: str):
    """wall paint: pick every image under any subfolder whose name contains `keyword`
    (e.g. 'worse_condtion'), case-insensitive."""
    out = []
    for cur_root, _, files in os.walk(root):
        if keyword.lower() in os.path.basename(cur_root).lower():
            for f in files:
                if f.lower().endswith(IMG_EXTS):
                    out.append(os.path.join(cur_root, f))
    return out


def collect_yolo_by_class_keyword(root: str, keywords: list[str]):
    """Stagnant Water and Wet Surface: YOLO format. classes.txt lists class
    names by line number = class id; each <name>.txt has 'class_id x y w h'
    lines. Include an image if any of its labeled boxes match a keyword."""
    classes_path = None
    for cur_root, _, files in os.walk(root):
        if "classes.txt" in files:
            classes_path = os.path.join(cur_root, "classes.txt")
            break
    if classes_path is None:
        raise FileNotFoundError(f"no classes.txt found under {root}")

    with open(classes_path) as f:
        class_names = [line.strip() for line in f if line.strip()]
    target_ids = {
        i for i, name in enumerate(class_names)
        if any(kw.lower() in name.lower() for kw in keywords)
    }
    print(f"classes.txt: {class_names}  -> matching ids for {keywords}: {target_ids}")

    label_dir = os.path.dirname(classes_path)
    out = []
    for txt_path in glob.glob(os.path.join(label_dir, "*.txt")):
        if os.path.basename(txt_path) == "classes.txt":
            continue
        with open(txt_path) as f:
            ids_in_file = {int(line.split()[0]) for line in f if line.strip()}
        if ids_in_file & target_ids:
            stem = os.path.splitext(txt_path)[0]
            for ext in IMG_EXTS:
                if os.path.exists(stem + ext):
                    out.append(stem + ext)
                    break
    return out


# --- S2DS concrete defect segmentation dataset ------------------------------
# Multi-class segmentation dataset. Masks are RGBA-color-coded per the
# dataset's own published legend (verified against this dataset's actual
# masks, not assumed):
#   background=(0,0,0,255)        crack=(255,255,255,255)
#   spalling=(255,0,0,255)        corrosion=(255,255,0,255)
#   efflorescence=(0,255,255,255) vegetation=(0,255,0,255)
#   control_point=(0,0,255,255)

def find_mask_pairs(root: str):
    """Returns [(image_path, mask_path), ...], trying two conventions:
    1. '<name>_lab.png' / '<name>_mask.png' next to '<name>.<ext>'
    2. sibling 'images/' and 'labels/' (or 'masks'/'annotations') folders
       with matching filenames (extension may differ, e.g. .jpg vs .png)
    Prints which convention matched so you can sanity-check it.
    """
    IMG_ONLY_EXTS = (".jpg", ".jpeg", ".png")

    # convention 1: suffix-based
    pairs = []
    for cur_root, _, files in os.walk(root):
        for f in files:
            for suffix in ("_lab", "_mask", "_label"):
                fl = f.lower()
                for ext in IMG_ONLY_EXTS:
                    if fl.endswith(f"{suffix}{ext}"):
                        base = f[: -(len(suffix) + len(ext))]
                        for img_ext in IMG_ONLY_EXTS:
                            img_path = os.path.join(cur_root, base + img_ext)
                            if os.path.exists(img_path):
                                pairs.append((img_path, os.path.join(cur_root, f)))
                                break
    if pairs:
        print(f"[mask pairing] matched via '_lab/_mask/_label' suffix: {len(pairs)} pairs")
        return pairs

    # convention 2: sibling images/ + labels/(masks/annotations/) folders
    img_dirs, label_dirs = [], []
    for cur_root, dirs, _ in os.walk(root):
        base = os.path.basename(cur_root).lower()
        if base in ("images", "image", "img", "imgs"):
            img_dirs.append(cur_root)
        elif base in ("labels", "label", "masks", "mask", "annotations", "gt", "groundtruth"):
            label_dirs.append(cur_root)

    for img_dir in img_dirs:
        for label_dir in label_dirs:
            if os.path.dirname(img_dir) != os.path.dirname(label_dir):
                continue
            label_files = {os.path.splitext(f)[0]: f for f in os.listdir(label_dir)}
            for f in os.listdir(img_dir):
                stem = os.path.splitext(f)[0]
                if stem in label_files:
                    pairs.append((os.path.join(img_dir, f), os.path.join(label_dir, label_files[stem])))

    if pairs:
        print(f"[mask pairing] matched via sibling images/labels folders: {len(pairs)} pairs")
        return pairs

    print(f"[!] [mask pairing] no image/mask pairs found under {root} with either "
          f"convention — inspect the folder tree manually and extend find_mask_pairs().")
    return []


def inspect_masks(root: str, n: int = 8):
    """Run this BEFORE trusting collect_by_mask_value. Prints the unique
    pixel values (and counts) found in a sample of masks, so you can see the
    real class encoding for this dataset."""
    import numpy as np
    from PIL import Image

    pairs = find_mask_pairs(root)
    print(f"found {len(pairs)} image/mask pairs under {root}")
    for img_path, mask_path in pairs[:n]:
        arr = np.array(Image.open(mask_path))
        vals, counts = np.unique(arr.reshape(-1, arr.shape[-1]) if arr.ndim == 3 else arr, axis=0, return_counts=True)
        print(os.path.basename(mask_path), "-> value:count", dict(zip(map(tuple, vals.tolist()) if arr.ndim == 3 else vals.tolist(), counts.tolist())))
    return pairs


def collect_by_mask_value(root: str, target_values: set, min_pixels: int = 200):
    """Keep only images whose mask contains at least `min_pixels` of a value
    in `target_values`, and where that target class outnumbers any other
    non-background value present. Background is treated as any all-zero
    value (0, (0,0,0), or (0,0,0,255) — RGBA with zero RGB) so RGBA masks
    like S2DS's don't get miscounted as "other" classes.
    Set `target_values` from what inspect_masks() printed for the class you
    want (e.g. spalling)."""
    import numpy as np
    from PIL import Image

    def is_background(v):
        if isinstance(v, tuple):
            return all(c == 0 for c in v[:3])  # ignore alpha channel
        return v == 0

    out = []
    for img_path, mask_path in find_mask_pairs(root):
        arr = np.array(Image.open(mask_path))
        if arr.ndim == 3:
            vals, counts = np.unique(arr.reshape(-1, arr.shape[-1]), axis=0, return_counts=True)
            value_counts = {tuple(v): c for v, c in zip(vals.tolist(), counts.tolist())}
        else:
            vals, counts = np.unique(arr, return_counts=True)
            value_counts = dict(zip(vals.tolist(), counts.tolist()))

        target_px = sum(c for v, c in value_counts.items() if v in target_values)
        other_px = sum(c for v, c in value_counts.items() if not is_background(v) and v not in target_values)
        if target_px >= min_pixels and target_px > other_px:
            out.append(img_path)
    return out

In [ ]:
# 2b) Mask pixel-value histogram for the S2DS spalling dataset.
# Confirmed color legend (downloaded from https://github.com/ben-z-original/s2ds
# colors/*.png and read directly, not guessed):
#   background=(0,0,0,255)  crack=(255,255,255,255)  spalling=(255,0,0,255)
#   corrosion=(255,255,0,255)  efflorescence=(0,255,255,255)
#   vegetation=(0,255,0,255)  control_point=(0,0,255,255)
# This matches the histogram we observed from this dataset's own masks.
_spalling_root_probe = find_dataset_root("s2ds-for-concrete-defects-segmentation")
inspect_masks(_spalling_root_probe, n=8)

SPALLING_MASK_VALUES = {(255, 0, 0, 255)}

In [ ]:
# 3) Collect (path, label) pairs per class using the collectors above.
# Must match the exact class order used in the InfraPulse backend
# (backend/app/ml/classifier.py::CLASSES) so the trained head's output
# indices line up correctly.
CLASSES = ["spalling", "stagnant_water", "paint_peeling", "cracked_tiles"]

TILES_ROOT = find_dataset_root("ceramic-tiles-defects-crackspotspinhole")
WATER_ROOT = find_dataset_root("stagnantwaterdata")
PAINT_ROOT = find_dataset_root("wall-paint")
SPALLING_ROOT = find_dataset_root("s2ds-for-concrete-defects-segmentation")

class_to_paths = {
    "cracked_tiles": collect_by_filename_prefix(TILES_ROOT, "crack"),
    "stagnant_water": collect_yolo_by_class_keyword(WATER_ROOT, ["water"]),
    "paint_peeling": collect_by_folder_keyword(PAINT_ROOT, "worse_condtion"),
    "spalling": collect_by_mask_value(SPALLING_ROOT, SPALLING_MASK_VALUES),
}

samples = []  # (path, class_index)
for cls_idx, cls in enumerate(CLASSES):
    for p in class_to_paths[cls]:
        samples.append((p, cls_idx))

from collections import Counter
counts = Counter(CLASSES[i] for _, i in samples)
print("per-class image counts:", dict(counts))
assert len(samples) > 0, "No images collected — check the *_ROOT paths against the inspect-cell output."
assert all(counts.get(c, 0) > 0 for c in CLASSES), (
    "Every class needs at least some images. If 'spalling' is 0, revisit "
    "SPALLING_MASK_VALUES using the histogram from the inspect_masks cell above."
)

In [ ]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

random.seed(42)
random.shuffle(samples)
split = int(0.85 * len(samples))
train_samples, val_samples = samples[:split], samples[split:]
print(f"train={len(train_samples)}  val={len(val_samples)}")

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class DefectDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

train_loader = DataLoader(DefectDataset(train_samples, train_tf), batch_size=16, shuffle=True)
val_loader = DataLoader(DefectDataset(val_samples, val_tf), batch_size=16, shuffle=False)

In [ ]:
# Model: frozen ImageNet MobileNetV2 backbone + small trainable head.
# Head architecture MUST match backend/app/ml/classifier.py::DefectHead
# exactly (1280 -> 128 -> len(CLASSES)) for the saved weights to load there.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
feature_extractor = backbone.features.to(device)
pool = nn.AdaptiveAvgPool2d(1)
for p in feature_extractor.parameters():
    p.requires_grad = False
feature_extractor.eval()

class DefectHead(nn.Module):
    def __init__(self, in_features=1280, num_classes=len(CLASSES)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)

head = DefectHead().to(device)
optimizer = torch.optim.Adam(head.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [ ]:
def embed(x):
    with torch.no_grad():
        feats = feature_extractor(x.to(device))
        feats = pool(feats).flatten(1)
    return feats

def evaluate(loader):
    head.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for x, y in loader:
            y = y.to(device)
            logits = head(embed(x))
            correct += (logits.argmax(1) == y).sum().item()
            n += y.size(0)
    head.train()
    return correct / max(n, 1)

EPOCHS = 25
for epoch in range(EPOCHS):
    total_loss, correct, n = 0.0, 0, 0
    for x, y in train_loader:
        y = y.to(device)
        feats = embed(x)
        logits = head(feats)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += x.size(0)

    val_acc = evaluate(val_loader)
    print(f"epoch {epoch+1}/{EPOCHS}  loss={total_loss/n:.4f}  train_acc={correct/n:.4f}  val_acc={val_acc:.4f}")

In [ ]:
# Confusion matrix on the val split — paste this into the documentation report.
import numpy as np

def confusion_matrix(loader, num_classes=len(CLASSES)):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    head.eval()
    with torch.no_grad():
        for x, y in loader:
            preds = head(embed(x)).argmax(1).cpu().numpy()
            for t, p in zip(y.numpy(), preds):
                cm[t, p] += 1
    head.train()
    return cm

cm = confusion_matrix(val_loader)
print("rows=true, cols=predicted, order:", CLASSES)
print(cm)

In [ ]:
# Save the trained head. Download this from the notebook's Output panel
# and copy it to backend/app/models_weights/head.pt in the InfraPulse repo.
out_path = "/kaggle/working/head.pt"
torch.save(head.state_dict(), out_path)
print(f"saved to {out_path}")